# Inheritance, Polymorphism, and Abstraction
### Continuing the `Friend` example → moving to `Shape`

**Today’s path**

1. Start from a class we already know: `Friend`.
2. Ask what a child class actually inherits.
3. Add specialized friend types and use `super()`.
4. Override behavior and see **polymorphism** emerge naturally.
5. Briefly connect inheritance to **encapsulation**.
6. Move to `Shape` to motivate **abstraction** and abstract methods.

> The goal is not to memorize syntax. We will keep asking: **what should be shared, what should vary, and what should be required?**

## 1. Starting point: a `Friend`

A `Friend` object stores information common to any kind of friend. We keep the class **concrete**: it makes sense to create an ordinary `Friend` object.

For this part, focus on the state and behavior that subclasses may reuse.

In [ ]:
from datetime import date


class Friend:
    """Represent a friend in an address book."""

    def __init__(self, first_name: str, last_name: str,
                 phone: str, dob: date) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.phone = phone
        self.dob = dob

    def introduce(self) -> str:
        """Return a one-line introduction."""
        return (
            "Hi, I'm " + self.first_name + " " + self.last_name
            + ". You can reach me at " + self.phone + "."
        )

    def describe_relationship(self) -> str:
        """Describe the relationship in a general way."""
        return self.first_name + " is my friend."

    def __str__(self) -> str:
        return self._first_name + " " + self._last_name

    def __lt__(self, other: "Friend") -> bool:
        return (self.last_name, self.first_name) < (other.last_name, other.first_name)

In [ ]:
friend = Friend("Sam", "Lee", "312-555-0100", date(1998, 4, 12))
print(friend)
print(friend.introduce())
print(friend.describe_relationship())

## 2. Inheritance: an **is-a** relationship

Suppose some friends need extra information:

- a `SchoolFriend` **is a** `Friend`, but we also want to remember the school;
- a `WorkFriend` **is a** `Friend`, but we also want to remember the company.

Inheritance lets us put the shared part in `Friend` instead of copying it into every specialized class.

```text
                  Friend
                 /      \
        SchoolFriend   WorkFriend
```

### What does a child class inherit?

A subclass can reuse the accessible behavior defined by its parent. An object of the subclass can therefore call inherited methods even when those methods are not rewritten in the subclass.

The subclass may also **add** new state/behavior or **override** inherited behavior.

### First experiment: inherit without adding anything

`SchoolFriend` contains no methods yet. What do you predict will happen?

In [ ]:
class SchoolFriend(Friend):
    pass


school_friend = SchoolFriend(
    "Maya", "Chen", "773-555-0101", date(1999, 8, 20)
)

print(school_friend)
print(school_friend.introduce())
print(school_friend.describe_relationship())

`SchoolFriend` did not define `introduce()`, `describe_relationship()`, or `__str__()`, but its objects can use them through inheritance.

**Important distinction:** the instance attributes such as `_first_name` are created when the inherited `__init__()` executes. The subclass inherits the initialization behavior that creates them.

## 3. Extending initialization with `super()`

Now a school friend needs one additional piece of state: `school`.

If we define a new `__init__()` in `SchoolFriend`, it **overrides** the inherited one. But we still need the parent initialization for the common friend data.

That is where `super()` is useful.

In [ ]:
class SchoolFriend(Friend):
    def __init__(self, first_name: str, last_name: str,
                 phone: str, dob: date, school: str) -> None:
        # Let Friend initialize the part that belongs to Friend.
        super().__init__(first_name, last_name, phone, dob)

        # Initialize only the new part here.
        self.school = school

In [ ]:
my_school_friend = SchoolFriend(
    "Maya", "Chen", "773-555-0101", date(1999, 8, 20), "Loyola"
)

print(my_school_friend._first_name)  # state initialized by Friend.__init__
print(my_school_friend._school)      # state added by SchoolFriend
print(my_school_friend.introduce()) # inherited behavior

### Why not repeat the parent's assignments?

We *could* assign `_first_name`, `_last_name`, `_phone`, and `_dob` again inside `SchoolFriend.__init__()`. But then the knowledge of how a `Friend` is initialized would be duplicated.

`super().__init__(...)` says: **initialize the Friend part of this object using the parent class's implementation.**

## 4. Overriding a method

The general `describe_relationship()` works, but a specialized friend can give a more useful description.

**Overriding** means defining a method in the child class with the same method name so that the child provides its own behavior.

In [ ]:
class SchoolFriend(Friend):
    def __init__(self, first_name: str, last_name: str,
                 phone: str, dob: date, school: str) -> None:
        super().__init__(first_name, last_name, phone, dob)
        self.school = school

    def describe_relationship(self) -> str:
        return self.first_name + " is my friend from " + self.school + "."


class WorkFriend(Friend):
    def __init__(self, first_name: str, last_name: str,
                 phone: str, dob: date, company: str) -> None:
        super().__init__(first_name, last_name, phone, dob)
        self.company = company

    def describe_relationship(self) -> str:
        return self.first_name + " is my friend from " + self.company + "."

In [ ]:
my_school_friend = SchoolFriend(
    "Maya", "Chen", "773-555-0101", date(1999, 8, 20), "Loyola"
)
my_work_friend = WorkFriend(
    "Omar", "Ali", "847-555-0102", date(1995, 2, 14), "Acme Corp"
)

print(my_school_friend.describe_relationship())
print(my_work_friend.describe_relationship())

# Both still reuse Friend.introduce().
print(my_school_friend.introduce())
print(my_work_friend.introduce())

## 5. Method lookup: which version runs?

When Python evaluates:

```python
my_school_friend.describe_relationship()
```

it starts with the object's class (`SchoolFriend`). If the method is found there, that version is used. If not, Python continues through the inheritance hierarchy.

So:

- `describe_relationship()` → found in `SchoolFriend` → **overridden version**;
- `introduce()` → not in `SchoolFriend` → found in `Friend` → **inherited version**.

You can inspect the method resolution order (MRO):

In [ ]:
print(SchoolFriend.mro())

## 6. What can be overridden?

Methods inherited from a parent can generally be overridden, including special/dunder methods.

For example, `WorkFriend` could provide a different `__str__()` if a work contact should display its company too.

In [ ]:
class WorkFriend(Friend):
    def __init__(self, first_name: str, last_name: str,
                 phone: str, dob: date, company: str) -> None:
        super().__init__(first_name, last_name, phone, dob)
        self.company = company

    def describe_relationship(self) -> str:
        return self.first_name + " is my friend from " + self.company + "."

    def __str__(self) -> str:
        return super().__str__() + " (" + self.company + ")"


my_work_friend = WorkFriend(
    "Omar", "Ali", "847-555-0102", date(1995, 2, 14), "Acme Corp"
)
print(my_work_friend)

Notice that `super()` is not limited to constructors. Here `super().__str__()` reuses the parent's implementation and extends its result.

## 7. Polymorphism appears naturally

**Polymorphism** means that the same operation can produce behavior appropriate to the actual object receiving it.

All three objects below can be treated as `Friend` objects. We call the **same method** each time, but Python chooses the implementation based on the actual object's class.

In [ ]:
friends: list[Friend] = [
    Friend("Sam", "Lee", "312-555-0100", date(1998, 4, 12)),
    SchoolFriend("Maya", "Chen", "773-555-0101", date(1999, 8, 20), "Loyola"),
    WorkFriend("Omar", "Ali", "847-555-0102", date(1995, 2, 14), "Acme Corp"),
]

for current_friend in friends:
    print(current_friend.describe_relationship())

### Inheritance and polymorphism are related, but not identical

Inheritance gives us the **relationship and shared implementation**. Overriding lets subclasses specialize behavior. Polymorphism is what lets client code simply call `describe_relationship()` without writing separate code for every friend type.

The caller asks for the behavior; the object determines which implementation runs.

In languages where types are enforced, polymorphism allows one variable, parameter, or list to accept objects of different classes, as long as they share a common parent type or interface, while still performing their own version of the same method.


## 8. Quick connection to encapsulation

Inheritance raises an important design question: **what should a subclass be allowed to access directly?**

If `Friend` protects an attribute behind a property, subclasses can use the public property instead of depending on its internal representation.

In [ ]:
class Friend:
    def __init__(self, first_name: str, phone: str) -> None:
        self.first_name = first_name
        self.__phone = phone

    @property
    def phone(self) -> str:
        """Provide controlled read access to the phone number."""
        return self.__phone

    @phone.setter
    def phone(self, value: str) -> None:
        """Control how the stored phone number may be changed."""
        if value == "":
            raise ValueError("Phone number cannot be empty.")
        self.__phone = value


class WorkFriend(Friend):
    def contact_line(self) -> str:
        # Use the public interface supplied by Friend.
        return self.first_name + ": " + self.phone

**Design idea:** inheritance should not mean “reach into everything the parent stores.” Encapsulation still matters. A subclass should prefer the interface its parent exposes.

Now we have seen a class hierarchy where the parent can stand on its own. But sometimes the parent represents an idea that is **too general to instantiate**. That motivates abstraction.

# 9. Abstraction: the `Shape` problem

Imagine that we want a collection of shapes. Every shape should have an `area()` operation.

Can we sensibly calculate the area of a generic `Shape`?

```text
                   Shape
                  /     \
              Circle   Rectangle
```

A `Circle` has an area formula. A `Rectangle` has an area formula. But **“shape” by itself does not tell us enough to calculate an area.**

We want the parent class to say:

> Every concrete shape **must provide** an `area()` method.

That is a good use for an **abstract class**.

## 10. Abstract base classes and abstract methods

An abstract class represents a general concept/contract, such as a Shape, that defines what its subclasses must be able to do without specifying exactly how they do it, ensuring that all subclasses follow a common structure.

Python provides `ABC` and `@abstractmethod` in the `abc` module.

- An **abstract base class (ABC)** can define common state and concrete behavior.
- An **abstract method** declares behavior that concrete subclasses are required to implement.
- A class that still has unimplemented abstract methods **cannot be instantiated**.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    """Abstract base class for geometric shapes."""

    def __init__(self, color: str) -> None:
        self.color = color

    def describe(self) -> str:
        """Concrete behavior shared by all shapes."""
        return "This shape is " + self.color + "."

    @abstractmethod
    def area(self) -> float:
        """Return the area of this shape."""
        pass

### Can we create a `Shape` object?

Predict what happens before running the next cell.

In [ ]:
# Uncomment to test:
# shape = Shape("blue")

Python prevents the instantiation because `Shape.area()` is abstract. This is intentional: there is no meaningful generic area calculation to use.

An abstract class is therefore not merely “a class we happen not to instantiate.” Python can **enforce** the abstraction.

## 11. Implementing the abstraction

A concrete subclass must provide the missing abstract behavior.

In [ ]:
from math import pi


class Circle(Shape):
    def __init__(self, color: str, radius: float) -> None:
        super().__init__(color)
        self.radius = radius

    def area(self) -> float:
        return pi * self.radius ** 2


class Rectangle(Shape):
    def __init__(self, color: str, width: float, height: float) -> None:
        super().__init__(color)
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

In [ ]:
circle = Circle("blue", 2.0)
rectangle = Rectangle("green", 3.0, 4.0)

print(circle.describe())
print(circle.area())

print(rectangle.describe())
print(rectangle.area())

## 12. What if a subclass forgets the abstract method?

Inheritance alone does not automatically make a subclass concrete.

In [ ]:
class MysteryShape(Shape):
    pass


# Uncomment to test:
# mystery = MysteryShape("purple")

`MysteryShape` still has no implementation of `area()`, so it remains abstract and cannot be instantiated.

This is one of the main benefits of an abstract method: the parent establishes a **contract** for its concrete subclasses.

## 13. Polymorphism again — now with an abstract contract

Because every concrete `Shape` must implement `area()`, code using shapes can rely on that operation without knowing each specific formula.

In [ ]:
shapes: list[Shape] = [
    Circle("blue", 2.0),
    Rectangle("green", 3.0, 4.0),
]

for shape in shapes:
    print(shape.describe(), "Area:", shape.area())

# 14. Put the ideas together

| Idea | Question it answers | In our examples |
|---|---|---|
| **Encapsulation** | What internal details should an object control or hide? | `Friend.phone` property |
| **Inheritance** | What state/behavior can related classes share? | `SchoolFriend` and `WorkFriend` reuse `Friend` |
| **Overriding** | How can a child specialize inherited behavior? | Different `describe_relationship()` methods |
| **Polymorphism** | Can the same call work across different object types? | `friend.describe_relationship()` / `shape.area()` |
| **Abstraction** | What behavior should every concrete subtype be required to provide? | `Shape.area()` |

### The key contrast

`Friend` is a useful object by itself, so keeping it **concrete** makes sense.

`Shape` represents a common concept, but a generic shape cannot supply a meaningful area formula. Making it **abstract** lets us share what is common while requiring subclasses to provide what varies.

## 15. Check your understanding

Before running code, discuss these questions:

1. If `SchoolFriend` does not define `introduce()`, which implementation is used?
2. Why do we call `super().__init__(...)` from a subclass constructor?
3. What is the difference between **inheriting** a method and **overriding** it?
4. In the `friends` loop, why can the same `describe_relationship()` call behave differently?
5. Why is `Friend` reasonable as a concrete class, while `Shape` is useful as an abstract class?
6. Can `Shape` contain a normal, implemented method such as `describe()` even though `Shape` is abstract?
7. What happens if a subclass of `Shape` does not implement `area()`?
8. What design benefit do we get from requiring every concrete `Shape` to implement `area()`?